#Initialization

In [0]:
%sql
USE CATALOG workspace

# Read Bronze table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df AS
SELECT * FROM workspace.bronze.erp_loc_a101

#Silver Transformations

##Trimming

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_trimmed AS
SELECT TRIM(cid)   AS cid,
       TRIM(cntry) AS cntry
FROM df

##Customer ID Cleanup

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_cid AS
SELECT REGEXP_REPLACE(cid, '-', '') AS cid,
       cntry
FROM df_trimmed

##Country Normalization

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_country AS
SELECT cid,
       CASE WHEN cntry = 'DE' THEN 'Germany'
            WHEN cntry IN ('US', 'USA') THEN 'United States'
            WHEN cntry = '' OR cntry IS NULL THEN 'n/a'
            ELSE cntry END AS cntry
FROM df_cid

## Renaming Columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_renamed AS
SELECT cid   AS customer_number,
       cntry AS country
FROM df_country

## Sanity checks of dataframe

In [0]:
%sql
SELECT * FROM df_renamed LIMIT 10

#Writing Silver Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.erp_customer_location USING DELTA AS
SELECT * FROM df_renamed

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10